In [1]:
# Carico la cartella one drive in cui ci sono:
# cartella video nutrition estimation / video annotation nutrition estimation
# cartella video nutrition change / video annotation nutrition change
# cartella video image nutrition estimation/ video annotation image estimation

# come ho scaricato i video?
# partendo dai file json ho lanciato lo script downloader sugli id estratti
# dei video scaricati ho fatto un samplig a N=128 FRAME
# caricato su one drive ed adesso linko da questo notebook
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Verifico la scheda video**
- Colab con GPU **L4** (24 GB di VRAM, circa 22 utilizzabili)
- serve Python 3.12 (vedi il controllo nella cella sotto)


In [ ]:
# Runtime: Runtime > Cambia tipo di runtime > GPU L4
import os, sys, subprocess
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # meno frammentazione della VRAM

!nvidia-smi
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "Nessuna GPU: Runtime > Cambia tipo di runtime > L4"
print("GPU:", gpu, "| Python:", sys.version.split()[0])
if "L4" not in gpu:
    print("ATTENZIONE: il notebook e' tarato su L4. Un'altra GPU va bene se ha almeno 24 GB.")
# Serve Python 3.12, come il baseline: su 3.13 tokenizers 0.19.1 non ha la wheel e la
# compilazione con Rust fallisce, quindi pip non installa transformers 4.40 ne' decord.
assert sys.version_info[:2] == (3, 12), (
    f"Python {sys.version.split()[0]}: serve 3.12 (come il baseline). Runtime > Cambia tipo di runtime > "
    "Versione runtime: sceglierne una con Python 3.12, poi riavviare la sessione.")

In [ ]:
# PyTorch 2.7.0, come il baseline. torchaudio va allineato: quello preinstallato da Colab
# e' compilato per un altro torch e non si potrebbe importare.
%pip install -q torch==2.7.0 torchvision==0.22.0 torchaudio==2.7.0 --index-url https://download.pytorch.org/whl/cu128

In [ ]:
# Repo ufficiale di VideoLLaMA 2, copia pulita fissata a un commit: se il repo cambia,
# il notebook usa comunque lo stesso codice del baseline.
import os, shutil, sys
VL2_DIR = "/content/VideoLLaMA2"
VL2_COMMIT = "c0bb03abf6b8a6b9a8dccac006fb4db5d4d9e414"   # ultimo commit del repo (gennaio 2025)

%cd /content
if os.path.exists(VL2_DIR):
    shutil.rmtree(VL2_DIR)          # toglie anche le patch di una sessione precedente
!git clone -q https://github.com/DAMO-NLP-SG/VideoLLaMA2.git {VL2_DIR}
!git -C {VL2_DIR} checkout -q {VL2_COMMIT}
!git -C {VL2_DIR} log -1 --format="VideoLLaMA2: commit %h del %cd"

if VL2_DIR not in sys.path:
    sys.path.insert(0, VL2_DIR)

In [ ]:
%cd /content/VideoLLaMA2

In [ ]:
%pip install -q --upgrade pip setuptools wheel   # servono a compilare tokenizers se Python e' 3.13

In [ ]:
# Versioni ufficiali di VideoLLaMA2 (pyproject.toml del repo):
#   transformers==4.40.0, tokenizers==0.19.1, accelerate==0.26.1, decord==0.6.0
# - tokenizers 0.19.1 ha la wheel per Python 3.12 ma non per 3.13: li' si compila con Rust (alcuni minuti).
# - huggingface_hub < 1.0 e' richiesto da transformers 4.40 (Colab ha la 1.x).
# - timm < 1.1: il repo importa timm.models.layers, deprecato.
# - peft NON si installa: VideoLLaMA2 non lo usa in inferenza, e una versione nuova
#   potrebbe trascinare un aggiornamento di transformers.
# Gli avvisi rossi di pip su gradio, sentence-transformers, peft o torchao sono attesi:
# sono pacchetti di Colab che questo notebook non importa.
#
# %pip non si ferma se l'installazione fallisce: se un pacchetto (es. tokenizers) non si
# compila, pip annulla TUTTO il comando e restano le versioni preinstallate di Colab.
# Per questo si usa pip() qui sotto, che si ferma e mostra l'errore vero.
import sys, os, subprocess

def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", *args],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("\n".join((r.stdout + r.stderr).splitlines()[-60:]))
        raise RuntimeError(f"pip install fallito: {' '.join(args)} (errore qui sopra)")

print("Python:", sys.version)

if sys.version_info >= (3, 13):
    print("Python 3.13: compilo tokenizers 0.19.1 con Rust (alcuni minuti)...")
    !curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal
    os.environ["PATH"] += ":/root/.cargo/bin"
    os.environ["PYO3_USE_ABI3_FORWARD_COMPATIBILITY"] = "1"   # pyo3 di tokenizers 0.19 non conosce ancora 3.13
    pip("tokenizers==0.19.1", "--no-binary", "tokenizers")
else:
    pip("tokenizers==0.19.1")

pip("transformers==4.40.0", "tokenizers==0.19.1", "accelerate==0.26.1", "huggingface_hub>=0.23,<1.0",
    "timm>=1.0.3,<1.1", "einops", "decord==0.6.0", "av", "imageio", "opencv-python")

# Controllo: se le versioni non sono quelle del baseline ci si ferma qui
from importlib.metadata import version, PackageNotFoundError
def _v(k):
    try:
        return version(k)
    except PackageNotFoundError:
        return "NON INSTALLATO"
_att = {"torch": "2.7.0", "transformers": "4.40.0", "tokenizers": "0.19.1",
        "accelerate": "0.26.1", "decord": "0.6.0"}
_vers = {k: _v(k) for k in _att}
print(_vers)
_err = {k: v for k, v in _vers.items() if not v.startswith(_att[k])}
assert not _err, f"versioni errate {_err}, attese {_att}: controllare l'output di pip sopra"

In [8]:
# NON esiste una env var "TRANSFORMERS_ATTN_IMPLEMENTATION" in transformers.
# L'attention eager va richiesta passando attn_implementation="eager" a model_init
print("Eager attention verra' impostato via attn_implementation='eager' in model_init")


Eager attention verra' impostato via attn_implementation='eager' in model_init


In [ ]:
import os
# Evita il lookup del secret HF_TOKEN di Colab che va in timeout e "appende" il download
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"   # download piu' stabile
os.environ.pop("HF_TOKEN", None)

import gc, sys, pathlib, torch
# Se torch era gia' stato importato prima dell'installazione, in memoria c'e' ancora quello vecchio
assert torch.__version__.startswith("2.7.0"), (
    f"torch {torch.__version__} in memoria, atteso 2.7.0: Runtime > Riavvia sessione, poi rieseguire dall'inizio")
assert torch.cuda.is_available(), "CUDA non disponibile: il runtime non ha la GPU"

# Libero la VRAM da tentativi precedenti
for name in ['model', 'processor', 'tokenizer']:
    if name in dir():
        exec(f"del {name}")
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f"VRAM libera: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GiB")

# -----------------------------------------------------------------------------
# PATCH encoder.py: il repo forza a mano config._attn_implementation="flash_attention_2"
# sulla CLIP/Siglip vision tower, ma questi modelli non supportano flash-attn -> ValueError.
# Sostituiamo con "eager" DIRETTAMENTE nel file sorgente, prima di importare videollama2.
# -----------------------------------------------------------------------------
_enc = pathlib.Path("/content/VideoLLaMA2/videollama2/model/encoder.py")
_txt = _enc.read_text()
if '"flash_attention_2"' in _txt:
    _enc.write_text(_txt.replace('"flash_attention_2"', '"eager"'))
    print("Patch applicata: flash_attention_2 -> eager in encoder.py")
else:
    print("Nessuna patch necessaria (flash_attention_2 non trovato in encoder.py)")

# Svuoto la cache dei moduli gia' importati, cosi' l'import qui sotto rilegge il file patchato
for _m in list(sys.modules):
    if _m.startswith("videollama2"):
        del sys.modules[_m]

# 1) Scarico lo snapshot VideoLLaMA2-7B (circa 16 GB)
from huggingface_hub import snapshot_download
model_path = snapshot_download("DAMO-NLP-SG/VideoLLaMA2-7B")
print("Modello scaricato in:", model_path)

# 2) Inizializzo passando il path LOCALE (niente altre chiamate di rete).
from videollama2 import model_init as videollama2_model_init

device = "cuda:0"
print("Inizio caricamento modello...")
model, processor, tokenizer = videollama2_model_init(
    model_path,
    device_map={"": device},
    attn_implementation="eager",   # forza eager sul modello linguistico
    use_flash_attn=False,
)

# imposto il modello in modalita' inferenza
model.eval()
print("Modello VideoLLaMA2-7B caricato con successo - dtype:", next(model.parameters()).dtype)
print("num_frames usati dal processor:", getattr(model.config, "num_frames", "default (8)"))
print(f"VRAM occupata: {torch.cuda.memory_allocated() / 1024**3:.1f} GiB "
      f"su {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB")

---
# Parte 1 — Riproduzione della baseline HD-EPIC (VideoLLaMA 2 7B)

Obiettivo: verificare il valore riportato nel paper HD-EPIC (Tab. 2, colonna **Nutrition**)
per VideoLLaMA 2 7B, cioe' **32.7%**, contro un baseline random del 20%.

 **L'accuracy di categoria e' la media delle accuracy per prototipo**, non l'accuracy
   aggregata sulle domande. 

## 1.1 Motore di valutazione

Prompt, parsing delle risposte e metriche stanno in un unico modulo, condiviso con il
notebook di LLaVA-Video. E' questo che rende confrontabili i due modelli: se ogni
notebook avesse il suo parser, le differenze di accuracy misurerebbero anche le
differenze fra i parser.

La cella scrive il modulo su disco e lo importa. La copia di riferimento sta in
`inference_folder/hdepic_eval.py` nel progetto.

In [10]:
%%writefile /content/hdepic_eval.py
"""
hdepic_eval - motore di valutazione per HD-EPIC VQA, categoria Nutrition.

Usato da entrambi i notebook di inferenza (VideoLLaMA2 e LLaVA-Video): tenere
prompt, parsing e metriche in un unico posto e' cio' che rende confrontabili i
risultati dei due modelli.

La parte specifica del modello e' tutta in `infer_fn(visual, prompt) -> str`.

Protocollo (fedele al paper HD-EPIC):
  - domande a 5 scelte, risposta come LETTERA (A-E)
  - accuracy di categoria = MEDIA DELLE ACCURACY PER PROTOTIPO
  - le domande con file mancanti o corrotti sono ESCLUSE, non contate sbagliate
  - un input con start_time/end_time e' la CLIP in quella finestra, non il video
    intero (README ufficiale: "one or more videos, one or more clips")
"""
import json, os, re, glob, unicodedata
import pandas as pd

VERSION = "1.1"
LETTERS = "ABCDEFGH"

# Valori di riferimento del paper HD-EPIC (Tab. 2, colonna Nutrition, % accuracy)
PAPER_NUTRITION = {
    "VideoLLaMA 2 7B":       32.7,
    "LLaVA-Video 7B":        38.7,
    "Llama 3.2 90B (blind)": 36.7,
    "Gemini Pro":            34.7,
    "Human":                 85.0,
    "Random":                20.0,
}

# task -> (file di annotation, modalita' di input nativa)
TASK_FILES = {
    "video_nutrition_estimation": ("nutrition_video_nutrition_estimation.json", "video"),
    "nutrition_change":           ("nutrition_nutrition_change.json",           "video"),
    "image_nutrition_estimation": ("nutrition_image_nutrition_estimation.json", "images"),
}
VIDEO_TASKS = [t for t, (_, m) in TASK_FILES.items() if m == "video"]

# Task i cui input sono clip [start_time, end_time] (da 1 a 27 s per nutrition_change).
# I fotogrammi vengono estratti a monte da preprocessing_folder/extract_clip_frames.py.
CLIP_TASKS = ["nutrition_change"]

# Task su cui ha senso l'esperimento Video Panels. Il paper applica i pannelli solo
# se il video dura piu' di gamma*C*alpha*beta = 1 s * 32 * 2 * 2 = 128 s; sotto
# soglia usa il campionamento normale. Le clip di nutrition_change (max 27 s) sono
# tutte sotto soglia, quindi il task e' escluso.
PANEL_TASKS = [t for t in VIDEO_TASKS if t not in CLIP_TASKS]


# ============================================================ indici dei file
def build_video_index(roots):
    """Indicizza TUTTI gli mp4 sotto `roots` -> {'sampled': {id: path}, 'full': {id: path}}.

    Indicizzare globalmente invece che cartella-per-task e' importante: lo stesso
    video puo' servire a piu' task ed essere stato scaricato in una sola cartella.
    """
    idx = {"sampled": {}, "full": {}}
    for root in roots:
        for p in glob.glob(os.path.join(root, "**", "*.mp4"), recursive=True):
            b = os.path.basename(p)
            if b.endswith("_sampled.mp4"):
                idx["sampled"].setdefault(b[:-len("_sampled.mp4")], p)
            else:
                idx["full"].setdefault(b[:-len(".mp4")], p)
    return idx


def build_panels_index(roots):
    """Indicizza le cartelle di pannelli -> {video_id: [path dei panel ordinati]}.

    Struttura attesa (output di mosaic.py): <root>/<video_id>/<video_id>_panel_0000.jpg
    """
    idx = {}
    for root in roots:
        if not os.path.isdir(root):
            continue
        for d in sorted(os.listdir(root)):
            full = os.path.join(root, d)
            if not os.path.isdir(full):
                continue
            panels = sorted(glob.glob(os.path.join(full, "*.jpg")) +
                            glob.glob(os.path.join(full, "*.jpeg")) +
                            glob.glob(os.path.join(full, "*.png")))
            if panels:
                idx.setdefault(d, panels)
    return idx


def resolve_video(video_id, index, prefer="sampled"):
    """-> (path, sorgente) con sorgente in {'sampled','full',None}."""
    order = ("sampled", "full") if prefer == "sampled" else ("full", "sampled")
    for src in order:
        if video_id in index[src]:
            return index[src][video_id], src
    return None, None


def hhmmss_to_sec(t):
    """'00:13:43.393' -> 823.393"""
    h, m, s = t.split(":")
    return int(h) * 3600 + int(m) * 60 + float(s)


def clip_name(video_id, start, end):
    """Nome della cartella di una clip (identico a extract_clip_frames.clip_name)."""
    return f"{video_id}_{int(round(start*1000)):09d}_{int(round(end*1000)):09d}"


def clip_frames(clip_cache, inp):
    """-> fotogrammi ordinati della clip di un input con start/end ([] se assenti)."""
    if not clip_cache:
        return []
    d = os.path.join(clip_cache, clip_name(inp["id"], hhmmss_to_sec(inp["start_time"]),
                                           hhmmss_to_sec(inp["end_time"])))
    return sorted(glob.glob(os.path.join(d, "frame_*.jpg")))


def _is_clip(inp):
    return "start_time" in inp and "end_time" in inp


def load_annotations(ann_dir, task):
    with open(os.path.join(ann_dir, TASK_FILES[task][0])) as f:
        return json.load(f)


# =================================================================== prompt
ANSWER_INSTRUCTION = "Answer with the option's letter from the given choices directly."

# Prompt di dominio opzionale: NON usarlo per il confronto col paper, che non ne ha uno.
SYSTEM_PROMPT = (
    "You are a precise video analysis assistant. Track ingredients, actions and "
    "weight measurements on digital scales to answer nutrition-related questions. "
    "Base your answer ONLY on the visual evidence provided."
)


def build_prompt(question, choices, system=None):
    """Prompt a lettere A-E. `system=None` = versione minimale, fedele al paper."""
    opts = "\n".join(f"{LETTERS[i]}. {c}" for i, c in enumerate(choices))
    head = f"{system}\n\n" if system else ""
    return f"{head}{question}\n{opts}\n{ANSWER_INSTRUCTION}"


NUTRIENT_RE = re.compile(r"\b(?:highest|higher)\s+(calories|carbs|fat|protein)\b", re.I)


def q_type_of(task, question):
    """Tipo di domanda: il nutriente chiesto, oppure il nome del prototipo."""
    m = NUTRIENT_RE.search(question)
    if m:
        return m.group(1).lower()
    return "nutrition_change" if task == "nutrition_change" else "unknown"


# ================================================================== parsing
def _norm(s):
    s = unicodedata.normalize("NFKD", s).lower()
    return re.sub(r"[^a-z0-9]+", " ", s).strip()


def parse_answer(response, choices):
    """Estrae l'indice della scelta -> (idx, metodo).

    A livelli, dal piu' stretto al piu' permissivo. Il metodo viene salvato nei
    risultati, cosi' e' sempre ispezionabile COME si e' arrivati a quell'indice.
    idx = -1 significa 'risposta non interpretabile'.
    """
    n = len(choices)
    valid = LETTERS[:n]
    s = (response or "").strip()
    if not s:
        return -1, "empty"

    # 1) inizia con la lettera: "C", "C.", "(C)", "C) spring greens"
    #    il lookahead evita di leggere la C di "Calories changed by..."
    m = re.match(r"^\W*\(?([A-Za-z])\)?(?=[\s.,:;)\-]|$)", s)
    if m and m.group(1).upper() in valid:
        return valid.index(m.group(1).upper()), "leading_letter"

    # 2) formula esplicita: "the answer is C", "Option B"
    m = re.search(r"\b(?:answer|option|choice)\s*(?:is|:)?\s*\(?([A-Za-z])\)?\b", s, re.I)
    if m and m.group(1).upper() in valid:
        return valid.index(m.group(1).upper()), "answer_is_letter"

    # 3) ha ricopiato il testo di una scelta invece della lettera
    ns = _norm(s)
    hits = [i for i, c in enumerate(choices) if _norm(c) and _norm(c) in ns]
    if len(hits) == 1:
        return hits[0], "choice_text"
    hits = [i for i, c in enumerate(choices) if ns and ns in _norm(c)]
    if len(hits) == 1:
        return hits[0], "choice_text_partial"

    # 4) una sola lettera valida isolata in tutto il testo
    cand = {x.upper() for x in re.findall(r"\b([A-Za-z])\b", s)} & set(valid)
    if len(cand) == 1:
        return valid.index(cand.pop()), "single_letter_in_text"

    # 5) ha risposto con una cifra nonostante il formato a lettere
    m = re.match(r"^\W*(\d)\b", s)
    if m and 1 <= int(m.group(1)) <= n:
        return int(m.group(1)) - 1, "digit_1indexed"

    return -1, "unparsed"


# ================================================= integrita' dati / pre-volo
def is_readable(path, cache=None):
    """Il file esiste ED e' decodificabile?

    Nel dataset scaricato ci sono mp4 da 0 byte e mp4 troncati (download
    interrotti): esistono ma non si aprono. Contarli come risposte sbagliate
    falserebbe l'accuracy, quindi vanno individuati PRIMA di girare.
    """
    if cache is not None and path in cache:
        return cache[path]
    ok = False
    if path and os.path.exists(path) and os.path.getsize(path) > 0:
        try:
            import cv2
            cap = cv2.VideoCapture(path)
            ok = bool(cap.read()[0])
            cap.release()
        except Exception:
            ok = False
    if cache is not None:
        cache[path] = ok
    return ok


def frame_path(frame_cache, video_id, sec):
    """Path del fotogramma in cache (stesso nome prodotto da extract_image_frames.py)."""
    return os.path.join(frame_cache, f"{video_id}_{int(round(sec*1000)):09d}.jpg")


def _required_inputs(item, task, index, panels_index, input_mode, prefer,
                     frame_cache=None, clip_cache=None):
    """-> [(video_id, risorsa)] necessaria a rispondere alla domanda.

    `risorsa` e' un path mp4, un path jpg gia' in cache, una lista di path di
    pannelli o di fotogrammi di una clip, o None se assente.
    """
    modality = TASK_FILES[task][1]
    out = []
    if modality == "images":
        # 5 fotogrammi a timestamp precisi: o sono gia' in cache, o serve il video
        # INTEGRALE. Mai ripiegare sul _sampled: ha 128 frame a 1 fps e il
        # timestamp finirebbe su un frame sbagliato senza nessun errore.
        for key in sorted(item["inputs"], key=lambda k: int(re.search(r"\d+", k).group())):
            inp = item["inputs"][key]
            cached = frame_cache and frame_path(frame_cache, inp["id"], hhmmss_to_sec(inp["time"]))
            if cached and os.path.exists(cached):
                out.append((inp["id"], cached))
            else:
                out.append((inp["id"], index["full"].get(inp["id"])))
    elif input_mode == "panels":
        for inp in item["inputs"].values():
            out.append((inp["id"], (panels_index or {}).get(inp["id"])))
    else:
        for inp in item["inputs"].values():
            if _is_clip(inp):
                # solo la finestra della domanda: mai il video intero o il _sampled,
                # che coprono minuti di video e quasi sempre mancano l'intervallo
                out.append((inp["id"], clip_frames(clip_cache, inp)))
            else:
                path, _ = resolve_video(inp["id"], index, prefer=prefer)
                out.append((inp["id"], path))
    return out


def usable_items(ann_dir, task, index, panels_index=None, input_mode="video",
                 prefer="sampled", cache=None, frame_cache=None, clip_cache=None):
    """Divide le domande in (utilizzabili, scartate-con-motivo)."""
    if input_mode == "panels" and task not in PANEL_TASKS:
        raise ValueError(f"{task}: Video Panels non applicabile (clip sotto la soglia "
                         f"di 128 s del paper). Usare tasks=H.PANEL_TASKS.")
    ann = load_annotations(ann_dir, task)
    ok, skipped = {}, {}
    for key, item in ann.items():
        bad = []
        for vid, res in _required_inputs(item, task, index, panels_index,
                                         input_mode, prefer, frame_cache, clip_cache):
            if isinstance(res, list):          # pannelli o fotogrammi di una clip
                if not res or any(os.path.getsize(p) == 0 for p in res):
                    bad.append(vid)
            elif isinstance(res, str) and res.endswith(".jpg"):  # frame in cache
                if os.path.getsize(res) == 0:
                    bad.append(vid)
            elif not is_readable(res, cache):  # video: deve aprirsi davvero
                bad.append(vid)
        if bad:
            skipped[key] = sorted(set(bad))
        else:
            ok[key] = item
    return ok, skipped


def preflight(ann_dir, index, tasks=None, panels_index=None, input_mode="video",
              prefer="sampled", cache=None, frame_cache=None, clip_cache=None):
    """Tabella di copertura reale. Da stampare SEMPRE prima di un run."""
    cache = {} if cache is None else cache
    rows, details = [], {}
    for task in (tasks or list(TASK_FILES)):
        ann = load_annotations(ann_dir, task)
        ok, skipped = usable_items(ann_dir, task, index, panels_index,
                                   input_mode, prefer, cache, frame_cache, clip_cache)
        details[task] = skipped
        rows.append({"task": task,
                     "input": ("frames" if TASK_FILES[task][1] == "images"
                               else "clip" if task in CLIP_TASKS else input_mode),
                     "domande": len(ann),
                     "utilizzabili": len(ok),
                     "scartate": len(skipped),
                     "file_inutilizzabili": len({v for ids in skipped.values() for v in ids})})
    df = pd.DataFrame(rows)
    df.loc[len(df)] = {"task": "TOTALE", "input": "", "domande": df["domande"].sum(),
                       "utilizzabili": df["utilizzabili"].sum(),
                       "scartate": df["scartate"].sum(), "file_inutilizzabili": ""}
    return df, details


# ======================================================== estrazione frame
def extract_frame(video_path, sec, out_path):
    """Salva il frame all'istante `sec`. Prova decord, poi ripiega su OpenCV.

    Serve per image_nutrition_estimation, dove ogni domanda mostra 5 fotogrammi
    presi da 5 video diversi a timestamp indicati nelle annotation.
    """
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    try:
        from decord import VideoReader
        from PIL import Image
        vr = VideoReader(video_path, num_threads=1)
        i = min(int(round(sec * vr.get_avg_fps())), len(vr) - 1)
        Image.fromarray(vr[i].asnumpy()).save(out_path, quality=95)
        return out_path
    except Exception:
        pass
    import cv2
    cap = cv2.VideoCapture(video_path)
    try:
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        i = int(round(sec * fps))
        if total:
            i = min(i, total - 1)
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ok, frame = cap.read()
        if not ok:                                  # seek per indice fallito
            cap.set(cv2.CAP_PROP_POS_MSEC, sec * 1000.0)
            ok, frame = cap.read()
        if not ok:
            raise RuntimeError(f"frame non leggibile a {sec:.3f}s in {video_path}")
        cv2.imwrite(out_path, frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
    finally:
        cap.release()
    return out_path


def _visual_input(item, task, index, panels_index, input_mode, prefer,
                  frame_extractor, frame_cache, clip_cache=None):
    """Prepara cio' che va passato al modello -> (input, etichetta sorgente)."""
    if TASK_FILES[task][1] == "images":
        paths = []
        for key in sorted(item["inputs"], key=lambda k: int(re.search(r"\d+", k).group())):
            inp = item["inputs"][key]
            sec = hhmmss_to_sec(inp["time"])
            out = frame_path(frame_cache, inp["id"], sec)
            if not os.path.exists(out):
                video = index["full"].get(inp["id"])   # solo integrale, vedi _required_inputs
                (frame_extractor or extract_frame)(video, sec, out)
            paths.append(out)
        return paths, "frames"
    inp = item["inputs"]["video 1"]
    vid = inp["id"]
    if _is_clip(inp):
        return clip_frames(clip_cache, inp), "clip"
    if input_mode == "panels":
        return panels_index[vid], "panels"
    path, src = resolve_video(vid, index, prefer=prefer)
    return path, src


# ================================================================ inferenza
def run_task(task, ann_dir, index, infer_fn, out_jsonl, panels_index=None,
             input_mode="video", prefer="sampled", system=None,
             frame_extractor=None, frame_cache=None, limit=None,
             verbose=True, readable_cache=None, clip_cache=None):
    """Inferenza su un task, con ripresa da dove si era interrotto.

    Ogni risposta viene scritta su `out_jsonl` subito: se Colab si disconnette,
    rilanciare la cella riprende senza rifare le domande gia' fatte.
    """
    ann, skipped = usable_items(ann_dir, task, index, panels_index,
                                input_mode, prefer, readable_cache, frame_cache, clip_cache)
    if frame_cache:
        os.makedirs(frame_cache, exist_ok=True)
    if verbose:
        print(f"  {len(ann)} domande utilizzabili"
              + (f", {len(skipped)} escluse (file mancanti o corrotti)" if skipped else ""))

    done = {}
    if os.path.exists(out_jsonl):
        with open(out_jsonl) as f:
            for line in f:
                r = json.loads(line)
                if r.get("error"):          # risposta fallita (es. CUDA OOM): va rifatta
                    done.pop(r["key"], None)
                else:
                    done[r["key"]] = r
        if verbose and done:
            print(f"  ripresa: {len(done)} risposte gia' presenti")

    results = []
    for n, (key, item) in enumerate(ann.items()):
        if limit and n >= limit:
            break
        if key in done:
            results.append(done[key])
            continue

        question, choices, correct = item["question"], item["choices"], item["correct_idx"]
        rec = {"key": key, "task": task, "input_mode": input_mode,
               "q_type": q_type_of(task, question),
               "video_id": list(item["inputs"].values())[0]["id"],
               "n_choices": len(choices), "correct_idx": correct}
        try:
            visual, src = _visual_input(item, task, index, panels_index, input_mode,
                                        prefer, frame_extractor, frame_cache, clip_cache)
            response = infer_fn(visual, build_prompt(question, choices, system=system))
            idx, method = parse_answer(response, choices)
            rec.update(source=src, response=response, predicted_idx=idx,
                       parse_method=method, error=None)
        except Exception as e:
            rec.update(source=None, response=None, predicted_idx=-1,
                       parse_method="error", error=f"{type(e).__name__}: {e}")

        rec["is_correct"] = bool(rec["predicted_idx"] == correct)
        with open(out_jsonl, "a") as f:
            f.write(json.dumps(rec) + "\n")
        results.append(rec)

        if verbose:
            mark = "ERR" if rec["error"] else ("OK " if rec["is_correct"] else "KO ")
            shown = (rec["response"] or rec["error"] or "")[:55]
            print(f"  [{n+1}/{len(ann)}] {mark} {key.split('_')[-1]:>3} | "
                  f"pred={rec['predicted_idx']} gt={correct} | "
                  f"{rec['parse_method']:<18} | {shown!r}")

    df = pd.DataFrame(results)
    df.attrs["skipped"] = skipped
    return df


def run_all(tasks, ann_dir, index, infer_fn, out_dir, tag, **kw):
    """Gira piu' task e concatena i risultati."""
    os.makedirs(out_dir, exist_ok=True)
    out, skipped = [], {}
    for task in tasks:
        print(f"\n{'='*70}\n  TASK: {task}   (run: {tag})\n{'='*70}")
        df = run_task(task, ann_dir, index, infer_fn,
                      os.path.join(out_dir, f"{tag}_{task}.jsonl"), **kw)
        skipped[task] = df.attrs.get("skipped", {})
        out.append(df)
    allr = pd.concat(out, ignore_index=True)
    allr.attrs["skipped"] = skipped
    return allr


# ================================================================= metriche
def accuracy_report(df, label="", paper_ref=None, ann_dir=None, show_bias=True):
    """Report allineato al protocollo del paper.

    HD-EPIC Tab. 2 e' la MEDIA DELLE ACCURACY PER PROTOTIPO, non l'accuracy
    aggregata sulle domande: con prototipi da 50/50/100 i due numeri non
    coincidono. Qui vengono stampati entrambi, cosi' il confronto e' esplicito.
    """
    out = {}
    per_task = df.groupby("task")["is_correct"].agg(corrette="sum", valutate="count")
    per_task["accuracy_%"] = (per_task["corrette"] / per_task["valutate"] * 100).round(1)
    if ann_dir:
        per_task["su_totale"] = [len(load_annotations(ann_dir, t)) for t in per_task.index]
    out["per_task"] = per_task
    out["nutrition_paper"] = round(per_task["accuracy_%"].mean(), 1)
    out["nutrition_pooled"] = round(df["is_correct"].mean() * 100, 1)

    print("=" * 70)
    print(f"  {label}   (n = {len(df)} domande valutate)")
    print("=" * 70)
    print(per_task.to_string())
    print(f"\n  Nutrition, metrica del paper (media per prototipo) : {out['nutrition_paper']}%")
    print(f"  Nutrition, accuracy aggregata sulle domande        : {out['nutrition_pooled']}%")
    print(f"  Baseline random                                    : 20.0%")
    if paper_ref in PAPER_NUTRITION:
        ref = PAPER_NUTRITION[paper_ref]
        print(f"  Paper, {paper_ref:<28}: {ref}%   "
              f"(delta {out['nutrition_paper'] - ref:+.1f})")

    inval = df[df["predicted_idx"] < 0]
    print(f"\n  Risposte non interpretate o in errore: {len(inval)}/{len(df)} "
          f"({len(inval)/max(len(df),1)*100:.1f}%)")
    print("  Come e' stata letta la risposta:")
    print("   ", df["parse_method"].value_counts().to_dict())
    out["parse_methods"] = df["parse_method"].value_counts()

    by_type = df.groupby("q_type")["is_correct"].agg(corrette="sum", valutate="count")
    by_type["accuracy_%"] = (by_type["corrette"] / by_type["valutate"] * 100).round(1)
    print("\n  Accuracy per tipo di domanda:")
    print(by_type.to_string())
    out["per_q_type"] = by_type

    if show_bias:
        print("\n  Bias posizionale (% risposte su ciascuna lettera, vs ground truth):")
        bias = {}
        for task, g in df.groupby("task"):
            n = int(g["n_choices"].max())
            pred = g[g["predicted_idx"] >= 0]["predicted_idx"].value_counts(normalize=True)
            gt = g["correct_idx"].value_counts(normalize=True)
            row = (pd.DataFrame({"pred": pred, "gt": gt})
                     .reindex(range(n)).fillna(0) * 100).round(0).astype(int)
            bias[task] = row["pred"]
            fmt = lambda c: "[" + " ".join(f"{v:3d}" for v in row[c]) + "]"
            print(f"    {task:<28} pred(A-E)={fmt('pred')}  gt(A-E)={fmt('gt')}")
        out["bias"] = pd.DataFrame(bias)
    return out


def compare_runs(runs, label="Confronto"):
    """Confronto fra piu' run: {nome: DataFrame} -> tabella accuracy per task."""
    rows = {}
    for name, df in runs.items():
        acc = (df.groupby("task")["is_correct"].mean() * 100).round(1)
        acc["NUTRITION (media prototipi)"] = round(acc.mean(), 1)
        rows[name] = acc
    tab = pd.DataFrame(rows)
    if len(runs) == 2:
        a, b = list(runs)
        tab["delta"] = (tab[b] - tab[a]).round(1)
    print("=" * 70)
    print(f"  {label}")
    print("=" * 70)
    print(tab.to_string())
    return tab

Writing /content/hdepic_eval.py


In [ ]:
# importo il modulo appena scritto (reload: se lo riscrivo, la sessione usa la versione nuova)
import sys, importlib
if "/content" not in sys.path:
    sys.path.insert(0, "/content")
import hdepic_eval as H
importlib.reload(H)

<module 'hdepic_eval' from '/content/hdepic_eval.py'>

## 1.2 Percorsi e indicizzazione dei file

I video sono sparsi in piu' cartelle (`videos_*_sampled`, `videos_*`, `videos_*_panels`) e
lo stesso video puo' servire a piu' prototipi pur essendo stato scaricato in una sola
cartella. Per questo si costruisce **un indice globale** `video_id -> path` invece di
cercare in una cartella fissa per task: cercando per cartella si perdono domande che
sarebbero perfettamente eseguibili.

In [ ]:
import os, glob

BASE = "/content/drive/MyDrive/Progetto Deep Learning 1"
ANN_DIR    = BASE                      # dove stanno i 3 json di annotation
OUT_DIR    = os.path.join(BASE, "risultati")
FRAME_DIR  = os.path.join(BASE, "frames_image_nutrition_estimation")
CLIP_DIR   = os.path.join(BASE, "frames_nutrition_change")   # clip di nutrition_change (extract_clip_frames.py)

# tutte le cartelle di video e di pannelli presenti nel progetto
VIDEO_ROOTS = sorted(d for d in glob.glob(os.path.join(BASE, "*"))
                     if os.path.isdir(d) and "panel" not in os.path.basename(d).lower()
                     and os.path.basename(d).lower().startswith(("video", "videos")))
PANEL_ROOTS = sorted(d for d in glob.glob(os.path.join(BASE, "*"))
                     if os.path.isdir(d) and "panel" in os.path.basename(d).lower())

print("cartelle video :", [os.path.basename(d) for d in VIDEO_ROOTS])
print("cartelle panels:", [os.path.basename(d) for d in PANEL_ROOTS])

VIDEO_INDEX = H.build_video_index(VIDEO_ROOTS)
PANEL_INDEX = H.build_panels_index(PANEL_ROOTS)
print(f"\nindicizzati: {len(VIDEO_INDEX['sampled'])} video campionati, "
      f"{len(VIDEO_INDEX['full'])} video integrali, {len(PANEL_INDEX)} cartelle di pannelli")
n_clip = len(glob.glob(os.path.join(CLIP_DIR, "*", "frame_000.jpg")))
print(f"clip di nutrition_change in {os.path.basename(CLIP_DIR)}: {n_clip} (attese 50)")

# controllo che le annotation ci siano davvero
for t, (fname, _) in H.TASK_FILES.items():
    p = os.path.join(ANN_DIR, fname)
    print(("  OK " if os.path.exists(p) else "  MANCA ") + fname)

## Quante domande sono davvero eseguibili

Nel dataset scaricato ci sono mp4 **da 0 byte** e mp4 **troncati**: esistono come file ma
non si aprono. Se non li si individua prima, ogni domanda che li usa produce un'eccezione
che finisce contata come risposta sbagliata, e l'accuracy risulta piu' bassa del vero.

Il pre-volo apre davvero ogni file e separa le domande eseguibili da quelle da escludere.
Le escluse **non entrano nel denominatore**: i risultati verranno riportati con il loro N.

In [ ]:
READABLE = {}   # cache: aprire i video da Drive e' lento, si controlla una volta sola

cov, dettagli = H.preflight(ANN_DIR, VIDEO_INDEX, cache=READABLE, frame_cache=FRAME_DIR,
                              clip_cache=CLIP_DIR)
print(cov.to_string(index=False))

rotti = sorted({v for task in dettagli.values() for ids in task.values() for v in ids})
if rotti:
    print(f"\nFile mancanti o non decodificabili ({len(rotti)}):")
    for v in rotti:
        print("   ", v)

## 1.4 Il modello

L'unica parte specifica di VideoLLaMA 2 e' la funzione `infer_fn`. Il `processor['video']`
del repo accetta sia il path di un mp4 sia una **lista di path di immagini** e in entrambi i
casi applica il preprocessing corretto per l'encoder CLIP (resize 336x336, normalizzazione
CLIP, padding `expand2square`). Costruire il tensore a mano con normalizzazioni sbagliate
produce output senza senso, quindi si passa sempre dal processor.

Da notare: il processor ricampiona a `model.config.num_frames` (8 di default) **solo** quando
riceve il path di un video. Una lista di immagini la carica tutta (fino a `MAX_FRAMES=32`),
quindi le liste lunghe (pannelli, fotogrammi delle clip di `nutrition_change`) vengono
sottocampionate uniformemente a 8 in `videollama2_infer`. Il campionamento a 128 frame fatto a
monte **non** arriva quindi tutto al modello.

In [ ]:
import numpy as np
from videollama2 import mm_infer

MAX_NEW_TOKENS = 32          # basta: la risposta attesa e' una lettera
USE_SYSTEM_PROMPT = None     # il paper non usa Systemp Prompt (se lo utilizzassimo le performance si alzano)
NUM_FRAMES = getattr(model.config, "num_frames", 8)   # elementi visivi per domanda (default 8)

def sottocampiona(paths, n=NUM_FRAMES):
    """Riduce una lista di immagini a n elementi uniformi (come _immagini in LLaVA).

    Serve perche' process_video ricampiona a num_frames SOLO quando riceve il path di un
    video: una lista di immagini la carica tutta e la tronca a MAX_FRAMES=32. Senza questo
    passaggio i 32 pannelli arriverebbero tutti al modello (4x i token visivi del baseline).
    Liste piu' corte di n (es. le 5 immagini di image_nutrition_estimation) restano invariate.
    """
    if len(paths) <= n:
        return paths
    sel = np.linspace(0, len(paths) - 1, n, dtype=int)
    return [paths[i] for i in sel]

def videollama2_infer(visual, prompt):
    """visual: path di un mp4, oppure lista di path di immagini (frame o pannelli)."""
    if isinstance(visual, list):
        visual = sottocampiona(visual)
    tensor = processor["video"](visual)
    # stesso numero di elementi visivi in ogni condizione (video, pannelli, immagini)
    assert tensor.shape[0] == NUM_FRAMES, f"attesi {NUM_FRAMES} elementi visivi, arrivati {tensor.shape[0]}"
    return mm_infer(tensor, prompt, model, tokenizer, modal="video",
                    do_sample=False,          # greedy: run deterministico e ripetibile
                    max_new_tokens=MAX_NEW_TOKENS)

print("num_frames usati dal processor:", NUM_FRAMES)
print("dtype:", next(model.parameters()).dtype)

## 1.6 Inferenza completa

I risultati vengono scritti su file **una domanda alla volta**: se Colab si disconnette,
basta rilanciare la cella e riparte da dove era arrivata invece di rifare tutto.

In [ ]:
baseline = H.run_all(list(H.TASK_FILES), ANN_DIR, VIDEO_INDEX, videollama2_infer,
                     out_dir=OUT_DIR, tag="videollama2_baseline_clip",   # nuovo tag: nutrition_change ora usa la clip
                     system=USE_SYSTEM_PROMPT,
                     frame_extractor=H.extract_frame, frame_cache=FRAME_DIR,
                     readable_cache=READABLE, clip_cache=CLIP_DIR)

print(f"\nCompletate {len(baseline)} domande")

In [17]:
rep_base = H.accuracy_report(baseline,
                             label="VideoLLaMA 2 7B - baseline (video)",
                             paper_ref="VideoLLaMA 2 7B",
                             ann_dir=ANN_DIR)#costruisco il report

  VideoLLaMA 2 7B - baseline (video)   (n = 200 domande valutate)
                            corrette  valutate  accuracy_%  su_totale
task                                                                 
image_nutrition_estimation        28       100        28.0        100
nutrition_change                   8        50        16.0         50
video_nutrition_estimation        33        50        66.0         50

  Nutrition, metrica del paper (media per prototipo) : 36.7%
  Nutrition, accuracy aggregata sulle domande        : 34.5%
  Baseline random                                    : 20.0%
  Paper, VideoLLaMA 2 7B             : 32.7%   (delta +4.0)

  Risposte non interpretate o in errore: 0/200 (0.0%)
  Come e' stata letta la risposta:
    {'leading_letter': 200}

  Accuracy per tipo di domanda:
                  corrette  valutate  accuracy_%
q_type                                          
calories                12        41        29.3
carbs                   20        35      

### Come leggere il bias posizionale

L'ultima tabella confronta **su quale lettera il modello mette le sue risposte** con la
distribuzione reale delle risposte corrette. Se il modello fosse solo impreciso, le due
distribuzioni si assomiglierebbero. Se invece concentra le risposte su A e B mentre il
ground truth e' uniforme, allora non sta rispondendo alla domanda: sta seguendo una
preferenza a priori sul formato. E' un modo per distinguere *"il modello sbaglia"* da
*"il modello non sta guardando il video"*, ed e' la spiegazione tipica di un'accuracy
**sotto** il 20% del random.

---
# Parte 2 — Video Panels (Visaul Prompting)

Secondo esperimento, basato su *Video Panels for Long Video Understanding. L'idea: invece di dare al modello 8 fotogrammi sparsi, gli si
danno 8 **immagini pannellate 2x2**, ognuna contenente 4 fotogrammi consecutivi. A parita'
di finestra di contesto il modello copre cosi' 32 fotogrammi invece di 8, scambiando
risoluzione spaziale con risoluzione temporale.

I pannelli sono stati generati a monte da `paper_mosaic_sampling/mosaic.py` (32 pannelli per
video, griglia 2x2, quindi 128 fotogrammi coperti).

**Ambito del confronto:** solo `video_nutrition_estimation` (`H.PANEL_TASKS`).
- `image_nutrition_estimation` resta fuori perche' il suo input sono 5 fotogrammi isolati da
  video diversi: non c'e' una sequenza temporale da impacchettare.
- `nutrition_change` resta fuori perche' il suo input e' una clip di 1-27 s. Il paper applica i
  pannelli solo sopra `gamma*C*alpha*beta` = 1 s x 128 = 128 s; sotto soglia ripiega sul
  campionamento normale, cioe' sullo stesso input del baseline.

In [ ]:
cov_p, dett_p = H.preflight(ANN_DIR, VIDEO_INDEX, tasks=H.PANEL_TASKS,
                            panels_index=PANEL_INDEX, input_mode="panels",
                            cache=READABLE)
print(cov_p.to_string(index=False))

# quanti pannelli per video, e quanti ne arrivano davvero al modello
if PANEL_INDEX:
    vid, pannelli = next(iter(PANEL_INDEX.items()))
    visti = len(sottocampiona(pannelli))
    print(f"\nesempio: {vid} -> {len(pannelli)} pannelli su disco")
    print(f"al modello ne arrivano {visti} (sottocampionati in videollama2_infer), "
          f"cioe' {visti * 4} fotogrammi originali contro gli {NUM_FRAMES} del baseline")

In [ ]:
panels_run = H.run_all(H.PANEL_TASKS, ANN_DIR, VIDEO_INDEX, videollama2_infer,
                       out_dir=OUT_DIR, tag="videollama2_panels_8",   # nuovo tag: il vecchio run usava 32 pannelli, non va ripreso
                       panels_index=PANEL_INDEX, input_mode="panels",
                       system=USE_SYSTEM_PROMPT, readable_cache=READABLE)

rep_pan = H.accuracy_report(panels_run,
                            label="VideoLLaMA 2 7B - Video Panels 2x2",
                            ann_dir=ANN_DIR)

## 2.1 Confronto

Il confronto va fatto **sulle stesse domande**: il baseline viene quindi ristretto a
`video_nutrition_estimation` e alle domande effettivamente valutate anche nel run con i pannelli.

In [ ]:
comuni = set(panels_run["key"]) & set(baseline["key"])
b = baseline[baseline["key"].isin(comuni)]
p = panels_run[panels_run["key"].isin(comuni)]
print(f"domande confrontate a parita' di input: {len(comuni)}\n")

tab = H.compare_runs({"baseline (video)": b, "panels 2x2": p},
                     "VideoLLaMA 2 7B: video liscio vs pannelli 2x2")

## 2.2 Salvataggio dei risultati

Tutto su CSV, una riga per domanda, con la risposta grezza del modello e il metodo con cui
e' stata interpretata: senza questi due campi un numero di accuracy non e' verificabile
da nessuno.

In [21]:
import pandas as pd

tutto = pd.concat([baseline.assign(run="baseline"),
                   panels_run.assign(run="panels")], ignore_index=True)
csv = os.path.join(OUT_DIR, "videollama2_risultati_completi.csv")
tutto.to_csv(csv, index=False)

riepilogo = pd.DataFrame({
    "run": ["baseline (video)", "panels 2x2"],
    "nutrition_media_prototipi_%": [rep_base["nutrition_paper"], rep_pan["nutrition_paper"]],
    "nutrition_aggregata_%":       [rep_base["nutrition_pooled"], rep_pan["nutrition_pooled"]],
    "domande_valutate":            [len(baseline), len(panels_run)],
})
riepilogo.to_csv(os.path.join(OUT_DIR, "videollama2_riepilogo.csv"), index=False)

print(riepilogo.to_string(index=False))
print(f"\nsalvato: {csv}")
print(f"paper, VideoLLaMA 2 7B su Nutrition: {H.PAPER_NUTRITION['VideoLLaMA 2 7B']}%  "
      f"| random: 20.0%")

             run  nutrition_media_prototipi_%  nutrition_aggregata_%  domande_valutate
baseline (video)                         36.7                   34.5               200
      panels 2x2                         41.2                   41.4                99

salvato: /content/drive/MyDrive/Progetto Deep Learning 1/risultati/videollama2_risultati_completi.csv
paper, VideoLLaMA 2 7B su Nutrition: 32.7%  | random: 20.0%
